# Classificação de Raças — Stanford Dogs Dataset + ConvNeXt-Tiny

**Objetivo:** Treinar e avaliar um classificador de raças de cães usando o Stanford Dogs Dataset (120 raças, ~20k imagens) com a arquitetura ConvNeXt-Tiny (pré-treinada ImageNet).

**Garantia:** Split estratificado 70/15/15 assegurando que **todas as 120 classes** apareçam nos conjuntos de treino, validação e teste.

**Bounding Boxes:** As anotações XML do dataset são usadas para recortar apenas a região do cão na imagem, eliminando fundo irrelevante.

---

## 1. Setup — Montagem do Drive e Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import csv
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from torch.amp import autocast, GradScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from collections import Counter

# Seed para reprodutibilidade
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.benchmark = True  # Auto-tune convolution algorithms for fixed input size

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuração de Caminhos (Google Drive)

In [ ]:
import subprocess, time

BASE_DIR    = '/content/drive/MyDrive/classification_exp'
OUTPUT_DIR  = os.path.join(BASE_DIR, 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Copia rapida para SSD local via tar (MUITO mais rapido que copytree) ---
LOCAL_DATA = '/content/local_data'
IMAGES_ROOT_DRIVE = os.path.join(BASE_DIR, 'Images')
ANNOT_ROOT_DRIVE  = os.path.join(BASE_DIR, 'Annotation')

assert os.path.isdir(IMAGES_ROOT_DRIVE), f"Pasta de imagens nao encontrada: {IMAGES_ROOT_DRIVE}"
assert os.path.isdir(ANNOT_ROOT_DRIVE),  f"Pasta de anotacoes nao encontrada: {ANNOT_ROOT_DRIVE}"

if not os.path.exists(os.path.join(LOCAL_DATA, 'Images')):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    print("Copiando dados do Drive para SSD local via tar (uma unica vez)...")
    t0 = time.time()
    # tar stream: copia tudo em um unico fluxo, muito mais rapido que copytree
    subprocess.run(
        f'tar -cf - -C "{BASE_DIR}" Images Annotation | tar -xf - -C "{LOCAL_DATA}"',
        shell=True, check=True
    )
    print(f"Copia concluida em {time.time()-t0:.1f}s")
else:
    print("Dados locais ja existem, pulando copia.")

IMAGES_ROOT = os.path.join(LOCAL_DATA, 'Images')
ANNOT_ROOT  = os.path.join(LOCAL_DATA, 'Annotation')

breed_dirs = sorted([d for d in os.listdir(IMAGES_ROOT)
                     if os.path.isdir(os.path.join(IMAGES_ROOT, d))])
print(f"Total de pastas (racas): {len(breed_dirs)}")
print(f"Primeiras 5: {breed_dirs[:5]}")
print(f"Resultados serao salvos em: {OUTPUT_DIR}")

## 3. Carregamento de Imagens, Labels e Bounding Boxes

In [ ]:
import os
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import xml.etree.ElementTree as ET
from tqdm import tqdm

def parse_annotation(annot_path):
    """Lê um XML e retorna (xmin, ymin, xmax, ymax) ou None."""
    try:
        # iterparse reduz overhead vs ET.parse em alguns casos
        xmin = ymin = xmax = ymax = None
        for event, elem in ET.iterparse(annot_path, events=("end",)):
            tag = elem.tag
            if tag == "xmin":
                xmin = int(elem.text)
            elif tag == "ymin":
                ymin = int(elem.text)
            elif tag == "xmax":
                xmax = int(elem.text)
            elif tag == "ymax":
                ymax = int(elem.text)

            # libera nós já processados (ajuda memória/velocidade)
            elem.clear()

        if None not in (xmin, ymin, xmax, ymax):
            return (xmin, ymin, xmax, ymax)
    except Exception:
        return None
    return None

def process_one_image(img_full, annot_full, breed_name):
    bbox = parse_annotation(annot_full) if annot_full is not None else None
    return img_full, bbox, breed_name

image_paths = []
bboxes = []
labels = []
breed_names_set = set()
bbox_found = 0

# Ajuste: 16~64 costuma ir bem no Colab; teste 32 primeiro
MAX_WORKERS = 32

print("Carregando imagens e anotacoes (paralelo)...")

futures = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    for breed_dir in tqdm(breed_dirs, desc="Racas processadas"):
        breed_name = breed_dir.split('-', 1)[1] if '-' in breed_dir else breed_dir
        breed_name = breed_name.replace('_', ' ')
        breed_names_set.add(breed_name)

        breed_img_path = os.path.join(IMAGES_ROOT, breed_dir)
        breed_annot_path = os.path.join(ANNOT_ROOT, breed_dir)

        # Lista imagens
        imgs = [f for f in os.listdir(breed_img_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        # Índice rápido de anotações existentes (evita exists() por arquivo)
        # (no dataset Oxford-IIIT Pet, as anotações não têm extensão)
        annot_set = set(os.listdir(breed_annot_path)) if os.path.isdir(breed_annot_path) else set()

        for img_file in imgs:
            img_full = os.path.join(breed_img_path, img_file)
            annot_name = os.path.splitext(img_file)[0]
            annot_full = os.path.join(breed_annot_path, annot_name) if annot_name in annot_set else None

            futures.append(ex.submit(process_one_image, img_full, annot_full, breed_name))

    for fut in tqdm(as_completed(futures), total=len(futures), desc="Parse XMLs"):
        img_full, bbox, breed_name = fut.result()
        image_paths.append(img_full)
        bboxes.append(bbox)
        labels.append(breed_name)
        if bbox is not None:
            bbox_found += 1

breed_names = sorted(list(breed_names_set))
label_to_idx = {name: idx for idx, name in enumerate(breed_names)}
idx_to_label = {idx: name for name, idx in label_to_idx.items()}
num_classes = len(breed_names)
labels_idx = [label_to_idx[l] for l in labels]

print(f"\nTotal de imagens: {len(image_paths)}")
print(f"Total de racas: {num_classes}")
print(f"Imagens com bounding box: {bbox_found} ({bbox_found/len(image_paths)*100:.1f}%)")

counts = Counter(labels)
min_breed = min(counts, key=counts.get)
max_breed = max(counts, key=counts.get)
print(f"Min amostras: {counts[min_breed]} ({min_breed})")
print(f"Max amostras: {counts[max_breed]} ({max_breed})")

## 4. Split Estratificado (70% Treino / 15% Validação / 15% Teste)

O uso de `stratify` garante que **todas as 120 classes** estejam representadas proporcionalmente nos três conjuntos.

In [ ]:
print("Realizando split estratificado...")
indices = list(range(len(image_paths)))

idx_train, idx_temp, _, _ = train_test_split(
    indices, labels_idx,
    test_size=0.30,
    stratify=labels_idx,
    random_state=RANDOM_SEED
)

temp_labels = [labels_idx[i] for i in idx_temp]

idx_val, idx_test, _, _ = train_test_split(
    idx_temp, temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=RANDOM_SEED
)

print("Separando dados por conjunto...")
X_train = [image_paths[i] for i in tqdm(idx_train, desc="Treino")]
y_train = [labels_idx[i] for i in idx_train]
bb_train = [bboxes[i] for i in idx_train]

X_val = [image_paths[i] for i in tqdm(idx_val, desc="Validacao")]
y_val = [labels_idx[i] for i in idx_val]
bb_val = [bboxes[i] for i in idx_val]

X_test = [image_paths[i] for i in tqdm(idx_test, desc="Teste")]
y_test = [labels_idx[i] for i in idx_test]
bb_test = [bboxes[i] for i in idx_test]

print(f"\nTreino:    {len(X_train)} imagens")
print(f"Validacao: {len(X_val)} imagens")
print(f"Teste:     {len(X_test)} imagens")

assert len(set(y_train)) == num_classes, f"Treino tem apenas {len(set(y_train))} classes!"
assert len(set(y_val)) == num_classes, f"Validacao tem apenas {len(set(y_val))} classes!"
assert len(set(y_test)) == num_classes, f"Teste tem apenas {len(set(y_test))} classes!"
print(f"Todas as {num_classes} classes presentes nos 3 conjuntos!")

## 5. Dataset com Bounding Box Crop + Data Augmentation

In [ ]:
IMG_SIZE = 224

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

import io

class DogBreedDataset(Dataset):
    """Dataset com pre-cache de imagens em RAM (bytes).
    Le cada arquivo JPEG uma unica vez e guarda os bytes na memoria.
    Isso elimina 100% do I/O de disco durante o treinamento.
    ~20k JPEGs ~ 1-2 GB em RAM — cabe tranquilo na A100 (83GB RAM).
    """
    def __init__(self, image_paths, labels, bboxes, transform=None, use_bbox=True):
        self.labels = labels
        self.bboxes = bboxes
        self.transform = transform
        self.use_bbox = use_bbox

        # Pre-cache: le todos os arquivos como bytes
        print(f"  Pre-carregando {len(image_paths)} imagens na RAM...")
        self.image_bytes = []
        for path in tqdm(image_paths, desc="  Cache RAM", leave=False):
            with open(path, 'rb') as f:
                self.image_bytes.append(f.read())
        size_mb = sum(len(b) for b in self.image_bytes) / (1024**2)
        print(f"  Cache: {len(self.image_bytes)} imagens, {size_mb:.0f} MB na RAM")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Abre da RAM, nao do disco
        image = Image.open(io.BytesIO(self.image_bytes[idx])).convert('RGB')
        label = self.labels[idx]

        if self.use_bbox and self.bboxes[idx] is not None:
            xmin, ymin, xmax, ymax = self.bboxes[idx]
            w, h = image.size
            margin_x = int((xmax - xmin) * 0.10)
            margin_y = int((ymax - ymin) * 0.10)
            xmin = max(0, xmin - margin_x)
            ymin = max(0, ymin - margin_y)
            xmax = min(w, xmax + margin_x)
            ymax = min(h, ymax + margin_y)
            image = image.crop((xmin, ymin, xmax, ymax))

        if self.transform:
            image = self.transform(image)

        return image, label


BATCH_SIZE = 256
VAL_BATCH_SIZE = 512

print("Criando datasets (com pre-cache na RAM)...")
train_dataset = DogBreedDataset(X_train, y_train, bb_train, train_transforms, use_bbox=True)
val_dataset   = DogBreedDataset(X_val, y_val, bb_val, eval_transforms, use_bbox=True)
test_dataset  = DogBreedDataset(X_test, y_test, bb_test, eval_transforms, use_bbox=True)

# Com dados na RAM, num_workers=4 e suficiente (evita overhead de IPC)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True,
                          prefetch_factor=2)
val_loader   = DataLoader(val_dataset, batch_size=VAL_BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True,
                          prefetch_factor=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True,
                          prefetch_factor=2)

print(f"Datasets criados! BATCH_SIZE={BATCH_SIZE}, VAL_BATCH_SIZE={VAL_BATCH_SIZE}")
print(f"Batches -> Treino: {len(train_loader)}, Val: {len(val_loader)}, Teste: {len(test_loader)}")

print("\nTestando carregamento de 1 batch...")
test_imgs, test_lbls = next(iter(train_loader))
print(f"Batch shape: {test_imgs.shape}, Labels shape: {test_lbls.shape}")
print("OK!")

## 6. Modelo ConvNeXt-Tiny (Fine-Tuning)

In [ ]:
def create_model(num_classes):
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)
    return model

model = create_model(num_classes).to(device)
model = torch.compile(model)  # PyTorch 2.x: fuses CUDA kernels for ~15-30% speedup
print(f"Modelo ConvNeXt-Tiny carregado com {num_classes} classes de saida.")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parametros Totais: {total_params:,}")
print(f"Parametros Treinaveis: {trainable_params:,}")

## 7. Treinamento com Early Stopping

In [ ]:
NUM_EPOCHS    = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 8

In [ ]:


criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

scaler = torch.amp.GradScaler('cuda')

train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_loss = float('inf')
best_val_acc  = 0.0
patience_counter = 0
best_epoch = 0

print(f"Iniciando treinamento: {NUM_EPOCHS} epocas, batch={BATCH_SIZE}/{VAL_BATCH_SIZE}, AMP=ON")
print("=" * 70)

for epoch in range(NUM_EPOCHS):
    # --- TREINO ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    train_pbar = tqdm(train_loader, desc=f"Ep {epoch+1:02d}/{NUM_EPOCHS} [Train]",
                      leave=False)
    for images, labels_batch in train_pbar:
        images, labels_batch = images.to(device, non_blocking=True), labels_batch.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)  # Otimizacao leve de memoria

        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels_batch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)

        train_pbar.set_postfix({
            'loss': f'{running_loss/total:.4f}',
            'acc': f'{correct/total:.4f}'
        })

    train_loss = running_loss / total
    train_acc  = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # --- VALIDACAO ---
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    val_pbar = tqdm(val_loader, desc=f"Ep {epoch+1:02d}/{NUM_EPOCHS} [Val]  ",
                    leave=False)
    with torch.no_grad():
        for images, labels_batch in val_pbar:
            images, labels_batch = images.to(device, non_blocking=True), labels_batch.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels_batch)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels_batch).sum().item()
            total += labels_batch.size(0)

            val_pbar.set_postfix({
                'loss': f'{running_loss/total:.4f}',
                'acc': f'{correct/total:.4f}'
            })

    val_loss = running_loss / total
    val_acc  = correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    scheduler.step()

    status = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc  = val_acc
        best_epoch    = epoch + 1
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_model_stanford.pth'))
        status = ' ** BEST **'
    else:
        patience_counter += 1
        status = f' (patience: {patience_counter}/{PATIENCE})'

    print(f"Ep {epoch+1:02d} | "
          f"Train L:{train_loss:.4f} A:{train_acc:.4f} | "
          f"Val L:{val_loss:.4f} A:{val_acc:.4f}{status}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping! Melhor epoca: {best_epoch}")
        break

print("=" * 70)
print(f"Treinamento finalizado!")
print(f"Melhor epoca: {best_epoch} | Val Loss: {best_val_loss:.4f} | Val Acc: {best_val_acc:.4f}")

## 8. Curvas de Treinamento (Loss e Acurácia)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].axvline(x=best_epoch-1, color='red', linestyle='--', alpha=0.7,
                label=f'Melhor Epoca ({best_epoch})')
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Loss')
axes[0].set_title('Curvas de Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(train_accs, label='Train Acc', linewidth=2)
axes[1].plot(val_accs, label='Val Acc', linewidth=2)
axes[1].axvline(x=best_epoch-1, color='red', linestyle='--', alpha=0.7,
                label=f'Melhor Epoca ({best_epoch})')
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Acuracia')
axes[1].set_title('Curvas de Acuracia'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Treinamento ConvNeXt-Tiny — Stanford Dogs (120 racas, com Bounding Box Crop)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'curvas_treinamento.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Avaliação Final no Conjunto de Teste

In [ ]:
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model_stanford.pth'),
                                 map_location=device))
model.eval()

all_preds = []
all_labels = []

print("Avaliando no conjunto de teste...")
with torch.no_grad():
    for images, labels_batch in tqdm(test_loader, desc="Teste"):
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_accuracy = accuracy_score(all_labels, all_preds)
print(f"\nAcuracia Global no Teste: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print("=" * 70)

In [ ]:
report = classification_report(
    all_labels, all_preds,
    labels=list(range(num_classes)),
    target_names=breed_names,
    zero_division=0,
    digits=3
)
print(report)

## 10. Salvar CSV com Métricas por Raça

In [ ]:
report_dict = classification_report(
    all_labels, all_preds,
    labels=list(range(num_classes)),
    target_names=breed_names,
    zero_division=0,
    output_dict=True
)

csv_path = os.path.join(OUTPUT_DIR, 'metricas_por_raca_stanford.csv')

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Raca', 'Precision', 'Recall', 'F1-Score', 'Support'])

    zero_support_count = 0
    for breed in breed_names:
        metrics = report_dict[breed]
        writer.writerow([
            breed,
            f"{metrics['precision']:.4f}",
            f"{metrics['recall']:.4f}",
            f"{metrics['f1-score']:.4f}",
            int(metrics['support'])
        ])
        if int(metrics['support']) == 0:
            zero_support_count += 1

    for avg_type in ['macro avg', 'weighted avg']:
        m = report_dict[avg_type]
        writer.writerow([
            avg_type,
            f"{m['precision']:.4f}",
            f"{m['recall']:.4f}",
            f"{m['f1-score']:.4f}",
            int(m['support'])
        ])

print(f"CSV salvo em: {csv_path}")
print(f"Racas com Support = 0: {zero_support_count}")
if zero_support_count == 0:
    print("TODAS as 120 racas possuem amostras no teste!")

## 11. Resultados Gerais (TXT)

In [ ]:
txt_path = os.path.join(OUTPUT_DIR, 'resultados_gerais_stanford.txt')

with open(txt_path, 'w', encoding='utf-8') as f:
    f.write("RESULTADOS — Classificacao de Racas (Stanford Dogs + ConvNeXt-Tiny)\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Dataset: Stanford Dogs Dataset\n")
    f.write(f"Total de Imagens: {len(image_paths)}\n")
    f.write(f"Total de Racas: {num_classes}\n")
    f.write(f"Bounding Box Crop: Sim (anotacoes XML Pascal VOC)\n")
    f.write(f"Split: 70% Treino ({len(X_train)}) / 15% Val ({len(X_val)}) / 15% Teste ({len(X_test)})\n")
    f.write(f"Split Estratificado: Sim (todas as classes em todos os conjuntos)\n\n")
    f.write(f"Modelo: ConvNeXt-Tiny (pre-treinado ImageNet, fine-tuned)\n")
    f.write(f"Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})\n")
    f.write(f"Scheduler: CosineAnnealingLR\n")
    f.write(f"Early Stopping: Patience={PATIENCE}\n")
    #f.write(f"Melhor Epoca: {best_epoch}\n\n")
    f.write(f"--- METRICAS NO CONJUNTO DE TESTE ---\n")
    f.write(f"Acuracia Global: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)\n")
    f.write(f"Precision (macro): {report_dict['macro avg']['precision']:.4f}\n")
    f.write(f"Recall (macro): {report_dict['macro avg']['recall']:.4f}\n")
    f.write(f"F1-Score (macro): {report_dict['macro avg']['f1-score']:.4f}\n")
    f.write(f"Precision (weighted): {report_dict['weighted avg']['precision']:.4f}\n")
    f.write(f"Recall (weighted): {report_dict['weighted avg']['recall']:.4f}\n")
    f.write(f"F1-Score (weighted): {report_dict['weighted avg']['f1-score']:.4f}\n")
    f.write(f"\n--- CLASSIFICATION REPORT COMPLETO ---\n\n")
    f.write(report)

print(f"Resultados salvos em: {txt_path}")

## 12. Distribuição de Acurácias por Raça

In [ ]:
acc_per_breed = []
for i, breed in enumerate(breed_names):
    mask = all_labels == i
    if mask.sum() > 0:
        acc = (all_preds[mask] == all_labels[mask]).mean()
    else:
        acc = 0.0
    acc_per_breed.append(acc)

acc_per_breed = np.array(acc_per_breed)

plt.figure(figsize=(12, 5))
plt.hist(acc_per_breed, bins=20, color='steelblue', edgecolor='black', alpha=0.8)
plt.axvline(x=acc_per_breed.mean(), color='red', linestyle='--', linewidth=2,
            label=f'Media: {acc_per_breed.mean():.3f}')
plt.xlabel('Acuracia por Raca', fontsize=12)
plt.ylabel('Numero de Racas', fontsize=12)
plt.title('Distribuicao das Acuracias Individuais — 120 Racas', fontsize=14, fontweight='bold')
plt.legend(fontsize=11); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'distribuicao_acuracias.png'), dpi=150, bbox_inches='tight')
plt.show()

sorted_idx = np.argsort(acc_per_breed)
print("\n--- TOP 10 MELHORES RACAS ---")
for i in sorted_idx[-10:][::-1]:
    print(f"  {breed_names[i]:40s} Acc: {acc_per_breed[i]:.4f}")

print("\n--- TOP 10 PIORES RACAS ---")
for i in sorted_idx[:10]:
    print(f"  {breed_names[i]:40s} Acc: {acc_per_breed[i]:.4f}")

## 13. Matriz de Confusão

In [ ]:
from matplotlib.colors import LogNorm

cm = confusion_matrix(all_labels, all_preds, labels=range(num_classes))

plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap='Blues', norm=LogNorm(), cbar=True,
            xticklabels=False, yticklabels=False)
plt.xlabel('Predito', fontsize=12)
plt.ylabel('Verdadeiro', fontsize=12)
plt.title('Matriz de Confusao — ConvNeXt-Tiny Stanford Dogs (120 racas)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'matriz_confusao.png'), dpi=150, bbox_inches='tight')
plt.show()

## 14. Exemplos Visuais de Predições (Acertos e Erros)

In [ ]:
def denormalize(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = img_tensor.cpu() * std + mean
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

N_SAMPLES = 16
indices = np.random.choice(len(test_dataset), N_SAMPLES, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
axes = axes.flatten()

for ax_idx, sample_idx in enumerate(indices):
    img, label = test_dataset[sample_idx]
    img_input = img.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_input)
        _, pred = torch.max(output, 1)
        pred = pred.item()

    pred_name = idx_to_label[pred]
    true_name = idx_to_label[label]
    correct = pred == label

    axes[ax_idx].imshow(denormalize(img))
    axes[ax_idx].axis('off')

    title = f"P: {pred_name}"
    if not correct:
        title += f"\nT: {true_name}"
        axes[ax_idx].set_title(title, fontsize=8, color='red', fontweight='bold')
    else:
        axes[ax_idx].set_title(title, fontsize=8, color='green')

plt.suptitle('Amostras de Predicoes (Verde=Correto, Vermelho=Erro)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'amostras_predicoes.png'), dpi=150, bbox_inches='tight')
plt.show()

## 15. Salvando Label Encoder (JSON)

In [ ]:
encoder_path = os.path.join(OUTPUT_DIR, 'label_encoder_stanford.json')
with open(encoder_path, 'w', encoding='utf-8') as f:
    json.dump({
        'label_to_idx': label_to_idx,
        'idx_to_label': {str(k): v for k, v in idx_to_label.items()},
        'breed_names': breed_names,
        'num_classes': num_classes
    }, f, ensure_ascii=False, indent=2)
print(f"Label encoder salvo em: {encoder_path}")

## 16. Resumo Final

In [ ]:
print("=" * 70)
print("RESUMO DO EXPERIMENTO")
print("=" * 70)
print(f"Dataset: Stanford Dogs — {len(image_paths)} imagens, {num_classes} racas")
print(f"Bounding Box Crop: Sim")
print(f"Modelo: ConvNeXt-Tiny (Fine-Tuned)")
print(f"Melhor Epoca: {best_epoch} (de {len(train_losses)} treinadas)")
print(f"Melhor Val Acc: {best_val_acc:.4f}")
print(f"\nTESTE FINAL:")
print(f"  Acuracia Global: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"  F1-Score (macro): {report_dict['macro avg']['f1-score']:.4f}")
print(f"  F1-Score (weighted): {report_dict['weighted avg']['f1-score']:.4f}")
print(f"\nArquivos salvos em: {OUTPUT_DIR}")
print(f"  - best_model_stanford.pth")
print(f"  - label_encoder_stanford.json")
print(f"  - metricas_por_raca_stanford.csv")
print(f"  - resultados_gerais_stanford.txt")
print(f"  - curvas_treinamento.png")
print(f"  - distribuicao_acuracias.png")
print(f"  - matriz_confusao.png")
print(f"  - amostras_predicoes.png")
print("=" * 70)